# 05. Выбор checkpoint детектора и временной ассоциации


`MyDrive/ML6/`.
Вычисления читают локальные кадры; результаты нового запуска пишутся в `runs/RUN_ID/`.


In [ ]:
import sys, os, subprocess, importlib.util
from pathlib import Path
IN_COLAB = 'google.colab' in sys.modules or Path('/content').is_dir()
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT = Path('/content/drive/MyDrive/ML6')
else:
    PROJECT = Path('D:/ML6')
RUN_ID = 'blue_aug_20261112_v2'  # одинаковый во всех 10 ноутбуках; старый run не меняем
BUNDLE = PROJECT / 'notebooks_colab_full'
assert (BUNDLE / 'tracking_lab.py').is_file(), BUNDLE
packages = {'scipy':'scipy', 'pandas':'pandas', 'cv2':'opencv-python-headless',
    'matplotlib':'matplotlib', 'tqdm':'tqdm', 'trackeval':'trackeval==1.3.0',
    'cryptography':'cryptography', 'imageio':'imageio', 'imageio_ffmpeg':'imageio-ffmpeg'}
missing = [p for m,p in packages.items() if importlib.util.find_spec(m) is None]
if missing:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *missing])
# Совместимую пару torch/torchvision предоставляет Colab; автоматически её не заменяем.
sys.path.insert(0, str(BUNDLE))
import tracking_lab as tl
assert Path(tl.__file__).resolve() == (BUNDLE / 'tracking_lab.py').resolve(), 'Перезапустите runtime: загружен старый модуль'
assert tl.SCHEMA == 4, 'Перезапустите runtime после обновления исходного кода'
import torch, pandas as pd, numpy as np, matplotlib.pyplot as plt
from IPython.display import display, Video, HTML, Image
lab = tl.Lab(PROJECT, RUN_ID)
torch.hub.set_dir(str(PROJECT / 'pretrained_cache'))
display(pd.Series(lab.environment(), name='Среда'))
print('Запуск:', RUN_ID, '| Результаты:', lab.root)

Mounted at /content/drive


,Среда
python,"3.13.16 (main, Oct 1 2026, 15:40:24) [GCC 13...."
packages,"{'torch': '2.14.1+cu130', 'torchvision': '0.29..."
device,cuda
gpu,NVIDIA L4


Запуск: blue_aug_20261112_v2 | Результаты: /content/drive/MyDrive/ML6/runs/blue_aug_20261112_v2


Восстановление кадров из архивов на Drive в `/content` с SHA256 каждого JPEG.
После сброса VM повторите эту ячейку. Checkpoint, manifest и архивы остаются на Drive.
Финальные видео здесь не загружаются для анализа.

In [ ]:
LOCAL = '/content/ml6_full_frames' if IN_COLAB else str(lab.path('local_frames'))
tl.cache_frames(lab, LOCAL, splits=('val',))
print('Validation-кэш:', lab.frame_cache_root)


Локальный кэш:   0%|          | 0/131 [00:00<?, ?архив/s]

Архив → /content:   0%|          | 0.00/29.7M [00:00<?, ?B/s]

Архив → /content:   0%|          | 0.00/39.3M [00:00<?, ?B/s]

Архив → /content:   0%|          | 0.00/13.3M [00:00<?, ?B/s]

Архив → /content:   0%|          | 0.00/16.1M [00:00<?, ?B/s]

Архив → /content:   0%|          | 0.00/21.8M [00:00<?, ?B/s]

... (4 outputs left out)

Архив → /content:   0%|          | 0.00/36.8M [00:00<?, ?B/s]

Архив → /content:   0%|          | 0.00/15.5M [00:00<?, ?B/s]

Архив → /content:   0%|          | 0.00/31.3M [00:00<?, ?B/s]

Validation-кэш: /content/ml6_full_frames


Калман использует реальный `dt` в секундах, Hungarian объединяет геометрию и
внешность. Сначала сопоставляются уверенные detections, затем слабые помогают
восстановить существующие треки по движению/IoU. Слабые detections не создают новые ID
и не обновляют feature bank. Банк обновляется только при высоком score и достаточном
разрыве до альтернатив в строке и столбце матрицы назначений. Сходство — медиана
по сохранённым признакам. GT/ignore-разметка в трекер не передаётся.

По три кандидата на вариант оцениваются на трёх фиксированных validation-видео
и отрезках до 180 последовательных позиций. Кэш detections/embedding повторно
используется между кандидатами; ассоциация проигрывается заново. HOTA/IDF1/MOTA/IDSW
считает TrackEval. Для суммарных метрик используется `combine_sequences`, не среднее
по видео.

Сначала оцениваем plain и natural с их best-весами. Для adversarial сравниваем два доступных checkpoint детектора: исходные best и last, каждый с тремя настройками трекера. ReID остаётся best из 04.

Критерий: среднее (HOTA+IDF1)/2 по clean, PGD, patch и ReID. Итоговая таблица показывает также чистое качество и recall. Это выбор на трёх validation-видео; гарантии качества на новых видео он не даёт.

Перед заменой detector best.pt сохраняются неизменяемые копии best_before_05_selection.pt и last_before_05_selection.pt вместе с SHA256. last.pt остаётся на месте. Выбор записывается в artifacts/detector_checkpoint_selection.json.нференс использует кэш с хешами фактических весов.


In [ ]:

import copy
import gc
import os
import shutil


def copy_verified_checkpoint(source, target):
    source, target = Path(source), Path(target)
    expected = tl.file_hash(source)
    if expected != tl.load_json(source.with_suffix('.sha256.json'))['sha256']:
        raise ValueError(f'Повреждён исходный checkpoint: {source}')
    target.parent.mkdir(parents=True, exist_ok=True)
    temporary = target.with_suffix('.selection.part')
    shutil.copy2(source, temporary)
    if tl.file_hash(temporary) != expected:
        raise ValueError('Ошибка копирования checkpoint')
    os.replace(temporary, target)
    tl.save_json(target.with_suffix('.sha256.json'), {'sha256': expected})
    return expected


def search_adversarial_checkpoints(lab):
    tl.ensure_trainable(lab)
    assert lab.device.type == 'cuda', 'Для этого этапа включите GPU'
    variant = 'adversarial'
    folder = tl.model_path(lab, 'detector', variant).parent
    journal = lab.path('artifacts/detector_checkpoint_selection.json')
    # Repair an interrupted installation before any consumer reads best.pt.
    if journal.exists():
        previous = tl.load_json(journal)
        if previous.get('phase') == 'installing':
            source = Path(previous['source'])
            if tl.file_hash(source) != previous['sha256']:
                raise ValueError('Изменился checkpoint незавершённого выбора')
            copy_verified_checkpoint(source, tl.model_path(lab, 'detector', variant))
            previous['phase'] = 'complete'
            tl.save_json(journal, previous)

    snapshots = {}
    for name in ('best', 'last'):
        target = folder / f'{name}_before_05_selection.pt'
        if not target.exists():
            copy_verified_checkpoint(tl.model_path(lab, 'detector', variant, name), target)
        snapshots[name] = target

    contract = tl.training_contract(lab, 'detector', variant)
    sequences = list(tl.group_sequences(lab.records('val')).values())[:lab.cfg['tracker_val_videos']]
    limit = lab.cfg['tracker_val_frames']
    records = [r for seq in sequences for r in
               (seq if len(seq) <= limit else seq[(len(seq)-limit)//2:][:limit])]
    if not records or any(r['split'] != 'val' for r in records):
        raise ValueError('Выбор допускает только непустую validation-выборку')
    reid = tl.load_model(lab, 'reid', variant)
    rows, conditions = [], []
    for name, path in snapshots.items():
        saved = tl.checked_checkpoint(path)
        if saved['contract'] != contract or saved['step'] != 0:
            raise ValueError('Нужен совместимый checkpoint завершённой эпохи')
        epoch = int(saved['epoch'])
        detector = tl.build_detector(lab)
        detector.load_state_dict(saved['model'])
        detector.eval()
        del saved
        weights = {'detector': tl.file_hash(path),
                   'reid': tl.file_hash(tl.model_path(lab, 'reid', variant))}
        print(f'Детектор {name}: эпоха {epoch}. Три настройки трекера; clean/PGD/patch/ReID.')
        for index, (age, gate, appearance) in enumerate([(2., .4, .5), (3., .45, .55), (5., .5, .6)]):
            cfg = copy.deepcopy(lab.cfg['tracker'])
            cfg.update(max_age_seconds=age, reid_gate=gate, appearance_weight=appearance)
            aggregate, _ = tl._evaluate(
                lab, detector, reid, records, ['clean', 'pgd', 'patch', 'reid'], cfg,
                variant, f'checkpoint_search_{name}_{index}', None, weights)
            score = float((aggregate.HOTA + aggregate.IDF1).mean()/2)
            if not np.isfinite(score):
                raise ValueError('Некорректная оценка checkpoint')
            row = dict(candidate=len(rows), score=score, config=cfg,
                       detector_checkpoint=name, detector_epoch=epoch,
                       detector_sha256=weights['detector'])
            rows.append(row)
            table = aggregate.assign(checkpoint=name, epoch=epoch, candidate=row['candidate'])
            conditions.append(table)
            display(table[['checkpoint', 'epoch', 'candidate', 'scenario', 'HOTA', 'IDF1', 'MOTA', 'recall', 'IDSW_total']])
            pd.concat(conditions, ignore_index=True).to_csv(
                lab.path('artifacts/detector_checkpoint_conditions.csv'), index=False)
            tl.save_json(lab.path('artifacts/detector_checkpoint_candidates.json'), rows)
        del detector
        gc.collect()
        torch.cuda.empty_cache()
    del reid
    gc.collect()
    torch.cuda.empty_cache()
    # Same selection rule as original 05: equal weight for the four scenarios.
    # On an exact tie, keep the earlier candidate (original best first).
    winner = max(rows, key=lambda row: row['score'])
    source = snapshots[winner['detector_checkpoint']]
    decision = dict(phase='installing', source=str(source), sha256=tl.file_hash(source),
                    winner=winner, records_contract=tl.digest(records),
                    rule='Mean (HOTA+IDF1)/2 over clean/PGD/patch/ReID on fixed validation segments')
    tl.save_json(journal, decision)
    copy_verified_checkpoint(source, tl.model_path(lab, 'detector', variant))
    tl.save_json(lab.path(f'configs/{variant}_tracker.json'), winner['config'])
    tl.save_json(lab.path(f'artifacts/{variant}_tracker_search.json'), rows)
    decision['phase'] = 'complete'
    tl.save_json(journal, decision)
    print('Выбран detector:', winner['detector_checkpoint'], '| эпоха:', winner['detector_epoch'])
    print('Оригинальные best/last сохранены:', folder)
    return pd.DataFrame(rows)


In [ ]:
for variant in ('plain', 'natural'):
    display(tl.tune_tracker(lab, variant))
    torch.cuda.empty_cache()

display(search_adversarial_checkpoints(lab))
display(tl.select_variant(lab))


uid_vid_00001.mp4 clean:   0%|          | 0/106 [00:00<?, ?it/s]

uid_vid_00001.mp4 pgd:   0%|          | 0/106 [00:00<?, ?it/s]

uid_vid_00001.mp4 patch:   0%|          | 0/106 [00:00<?, ?it/s]

uid_vid_00001.mp4 reid:   0%|          | 0/106 [00:00<?, ?it/s]

uid_vid_00012.mp4 clean:   0%|          | 0/79 [00:00<?, ?it/s]

uid_vid_00012.mp4 pgd:   0%|          | 0/79 [00:00<?, ?it/s]

... (147 outputs left out)

,checkpoint,epoch,candidate,scenario,HOTA,IDF1,MOTA,recall,IDSW_total
0,last,6,5,clean,0.296158,0.285200,-0.026793,0.766282,1013
1,last,6,5,pgd,0.175746,0.162593,-0.810800,0.539571,706
2,last,6,5,patch,0.286956,0.275777,-0.038747,0.762366,1006
3,last,6,5,reid,0.298171,0.288126,-0.027824,0.764839,1021


Выбран detector: best | эпоха: 2
Оригинальные best/last сохранены: /content/drive/MyDrive/ML6/runs/blue_aug_20261112_v2/checkpoints/adversarial/detector


,candidate,score,config,detector_checkpoint,detector_epoch,detector_sha256
0,0,0.265067,"{'max_age_seconds': 2.0, 'min_hits': 2, 'reid_...",best,2,72d155dc56d99101b5a8b6783cf8fe2169f07787e3424f...
1,1,0.260724,"{'max_age_seconds': 3.0, 'min_hits': 2, 'reid_...",best,2,72d155dc56d99101b5a8b6783cf8fe2169f07787e3424f...
2,2,0.261668,"{'max_age_seconds': 5.0, 'min_hits': 2, 'reid_...",best,2,72d155dc56d99101b5a8b6783cf8fe2169f07787e3424f...
3,3,0.256866,"{'max_age_seconds': 2.0, 'min_hits': 2, 'reid_...",last,6,e5585353b36d9835339c0989a8840140e6453146539ab7...
4,4,0.258183,"{'max_age_seconds': 3.0, 'min_hits': 2, 'reid_...",last,6,e5585353b36d9835339c0989a8840140e6453146539ab7...
5,5,0.258591,"{'max_age_seconds': 5.0, 'min_hits': 2, 'reid_...",last,6,e5585353b36d9835339c0989a8840140e6453146539ab7...


,variant,validation_score
0,plain,0.273246
1,natural,0.282458
2,adversarial,0.265067


Блокнот подобрал настройки трекера и выбрал natural для дальнейшего использования.
- Общая оценка: natural — 0,282, plain — 0,273, adversarial — 0,265.
- Для adversarial веса второй эпохи оказались лучше шестой: выигрыш под PGD не компенсировал ухудшение в остальных условиях.
- Подбор настроек дал небольшой прирост. Надёжная защита от атак не достигнута.
- Выбор сделан на трёх validation-видео; качество на новых видео ещё нужно провери

Нежданчик